# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [3]:
# Load the libraries as required.
import os
import sys
sys.path.append(os.getenv('SRC_DIR'))
import pandas as pd
import numpy as np
import os

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn import linear_model
from sklearn.ensemble import GradientBoostingRegressor

In [4]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [5]:
X = fires_dt.drop(columns='area')

In [6]:
Y = fires_dt['area']

In [7]:
#scoring = ['neg_log_loss', 'roc_auc', 'f1', 'accuracy', 'precision', 'recall','neg_mean_squared_error']
scoring = ['neg_mean_absolute_error','neg_mean_squared_error']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [8]:
preproc1 = ColumnTransformer(
    transformers=[
        ('numeric_transfomer', StandardScaler(), ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain'] ),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist'), ['month', 'day']), 
    ], remainder='passthrough'
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [9]:
from sklearn.preprocessing import FunctionTransformer
#from sklearn.preprocessing import PowerTransformer
pt = FunctionTransformer()

preproc2 = ColumnTransformer(
    transformers=[
        ('numeric_transfomer', StandardScaler(), ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain'] ),
        ('Power_transfomer', pt, ['temp'] ),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist'), ['month', 'day']), 
    ], remainder='passthrough'
)


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [10]:
# Pipeline A = preproc1 + baseline
pipe_knr = Pipeline([
    ('preprocess', preproc1),
    ('knr', KNeighborsRegressor())
    #('knr', linear_model.Ridge())
])
pipe_knr




Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr', KNeighborsRegressor())])

In [11]:
# Pipeline B = preproc2 + baseline
pipe_knr2 = Pipeline([
    ('preprocess2', preproc2),
    ('knr2', KNeighborsRegressor())
])
pipe_knr2

Pipeline(steps=[('preprocess2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('Power_transfomer',
                                                  FunctionTransformer(),
                                                  ['temp']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr2', KNeighborsRegressor())])

In [12]:
# Pipeline C = preproc1 + advanced model

pipe_knr3 = Pipeline([
    ('preprocess3', preproc1),
    ('knr3', GradientBoostingRegressor())
])
pipe_knr3

Pipeline(steps=[('preprocess3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr3', GradientBoostingRegressor())])

In [13]:
# Pipeline D = preproc2 + advanced model

pipe_knr4 = Pipeline([
    ('preprocess4', preproc2),
    ('knr4', GradientBoostingRegressor())
    ])
pipe_knr4

Pipeline(steps=[('preprocess4',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('Power_transfomer',
                                                  FunctionTransformer(),
                                                  ['temp']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr4', GradientBoostingRegressor())])

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [14]:
param_grid1 = {
    'knr__n_neighbors': [3, 5, 7,9],
    'knr__p': [1,2]
    }

grid_cv1 = GridSearchCV(
    estimator=pipe_knr, 
    param_grid=param_grid1, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")

grid_cv1.fit(X_train, Y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transfomer',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('knr', KNeighborsRegressor())]),
             param_grid={'knr__n_neighbors': [3, 5, 7, 9], 'knr__p': [1, 2]},
             refit='neg_mean_squared_error',
             scoring=['neg_mean_absolute_error', 'neg_mean_squared_error'])

In [15]:
res1 = grid_cv1.cv_results_
res1 = pd.DataFrame(res1)
res1

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr__n_neighbors,param_knr__p,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
0,0.034303,0.009217,1.375272,2.702746,3,1,"{'knr__n_neighbors': 3, 'knr__p': 1}",-16.449157,-15.533936,-17.184779,...,3.889249,5,-1873.187968,-1218.214595,-1451.674325,-7776.906675,-1446.432067,-2753.283126,2520.707901,7
1,0.013717,0.005219,0.031726,0.005747,3,2,"{'knr__n_neighbors': 3, 'knr__p': 2}",-21.212851,-16.434137,-17.974498,...,3.747099,7,-2813.326232,-1608.239108,-2102.398837,-7812.569202,-1035.386921,-3074.384060,2440.098164,8
2,0.036093,0.017486,0.036906,0.008488,5,1,"{'knr__n_neighbors': 5, 'knr__p': 1}",-20.471060,-18.261687,-16.213084,...,4.019720,8,-2377.314564,-1346.299998,-1161.860454,-7756.846647,-1021.164247,-2732.697182,2556.847466,6
3,0.035762,0.011562,0.038903,0.007452,5,2,"{'knr__n_neighbors': 5, 'knr__p': 2}",-17.999157,-17.034410,-16.389687,...,3.851314,4,-1993.330721,-1291.160163,-1171.387382,-7689.458221,-645.651546,-2558.197607,2601.359991,4
4,0.042213,0.010977,0.033073,0.017097,7,1,"{'knr__n_neighbors': 7, 'knr__p': 1}",-20.923804,-17.772117,-17.131962,...,4.227390,6,-2477.958047,-981.888595,-1023.938486,-7603.609229,-775.558697,-2572.590611,2587.588981,5
5,0.021522,0.005565,0.015161,0.008037,7,2,"{'knr__n_neighbors': 7, 'knr__p': 2}",-18.251222,-15.238864,-15.954768,...,4.018582,2,-1910.257821,-772.865721,-984.583811,-7401.428245,-571.227434,-2328.072606,2577.737077,2
6,0.011699,0.004189,0.017879,0.008802,9,1,"{'knr__n_neighbors': 9, 'knr__p': 1}",-18.646345,-15.658527,-16.957323,...,3.998131,3,-1901.599844,-674.770463,-939.734348,-7471.110576,-684.242932,-2334.291633,2607.511928,3
7,0.026787,0.018139,0.030989,0.014040,9,2,"{'knr__n_neighbors': 9, 'knr__p': 2}",-17.451071,-15.100161,-15.519344,...,3.896534,1,-1729.479837,-641.062780,-884.707057,-7357.961542,-656.778963,-2253.998036,2582.757809,1


In [16]:
res1 = grid_cv1.cv_results_
res1 = pd.DataFrame(res1)
##res1
res1.columns

res1[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr__n_neighbors', 'param_knr__p', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error', 'split0_test_recall',
       'split1_test_recall', 'split2_test_recall', 'split3_test_recall',
       'split4_test_recall', 'mean_test_recall', 'std_test_recall',
       'rank_test_recall', 'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error']].sort_values('rank_test_neg_mean_squared_error')

#Note that best param {knr__n_neighbors: 9,knr__p:2}, Avg mean = -2253.9


KeyError: "['split0_test_recall', 'split1_test_recall', 'split2_test_recall', 'split3_test_recall', 'split4_test_recall', 'mean_test_recall', 'std_test_recall', 'rank_test_recall'] not in index"

In [17]:
Best_param1 = grid_cv1.best_params_
#Best_param1 = pd.DataFrame(Best_param1)
print(Best_param1)
print(grid_cv1.best_estimator_)

{'knr__n_neighbors': 9, 'knr__p': 2}
Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr', KNeighborsRegressor(n_neighbors=9))])


In [18]:
param_grid2 = {
    'knr2__n_neighbors': [3, 5, 7,9],
    'knr2__p': [1,2]
    }


grid_cv2 = GridSearchCV(
    estimator=pipe_knr2, 
    param_grid=param_grid2, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")

grid_cv2.fit(X_train, Y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transfomer',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('Power_transfomer',
                                                                         FunctionTransformer(),
                                                                         ['temp']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('knr2', KNeighborsRegressor())]),
             param_grid={'knr2__n_neighbors': [3, 5, 7, 9], 'knr2__p': [1, 2]},
             refit='neg_mean_squared_error',
             scoring=['neg_mean_absolute_error', 'neg_mean_squared_error'])

In [19]:
res2 = grid_cv2.cv_results_
res2 = pd.DataFrame(res2)
res2

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr2__n_neighbors,param_knr2__p,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
0,0.034416,0.006019,0.033382,0.012353,3,1,"{'knr2__n_neighbors': 3, 'knr2__p': 1}",-17.627349,-19.975984,-19.183454,...,3.566867,8,-1939.175732,-2020.643700,-1741.021132,-7663.872051,-978.036767,-2868.549876,2425.825914,8
1,0.027933,0.007225,0.017409,0.004702,3,2,"{'knr2__n_neighbors': 3, 'knr2__p': 2}",-14.987108,-20.503936,-13.972450,...,3.949761,4,-1833.918297,-2112.196932,-856.201301,-7236.719594,-1079.006904,-2623.608606,2352.677992,6
2,0.020263,0.006340,0.018963,0.004488,5,1,"{'knr2__n_neighbors': 5, 'knr2__p': 1}",-19.651831,-18.097277,-18.797783,...,3.516038,7,-2363.566108,-1200.156069,-1393.447609,-7513.018880,-700.535121,-2634.144757,2498.444504,7
3,0.031900,0.003400,0.012198,0.002583,5,2,"{'knr2__n_neighbors': 5, 'knr2__p': 2}",-16.396289,-16.562771,-14.194145,...,3.114068,3,-1949.759575,-966.609921,-847.609792,-7318.201060,-1057.548943,-2427.945858,2476.053738,4
4,0.027963,0.004994,0.016102,0.001372,7,1,"{'knr2__n_neighbors': 7, 'knr2__p': 1}",-18.786902,-16.471377,-18.522444,...,3.747471,6,-2111.917956,-749.030636,-1331.544077,-7317.116646,-849.013322,-2471.724527,2470.148292,5
5,0.019323,0.002553,0.024370,0.002827,7,2,"{'knr2__n_neighbors': 7, 'knr2__p': 2}",-14.828640,-14.400258,-16.855938,...,3.494109,2,-1555.695932,-650.224353,-1157.582064,-6942.610238,-821.009093,-2225.424336,2378.835986,2
6,0.021349,0.009675,0.022442,0.009362,9,1,"{'knr2__n_neighbors': 9, 'knr2__p': 1}",-17.043467,-17.011486,-18.010067,...,3.755056,5,-1607.519929,-900.159146,-1138.703982,-7017.940723,-745.236204,-2281.911997,2385.888126,3
7,0.020122,0.008718,0.024358,0.005590,9,2,"{'knr2__n_neighbors': 9, 'knr2__p': 2}",-14.929786,-14.416827,-16.194659,...,3.531032,1,-1539.203920,-633.054250,-998.444498,-6948.007445,-761.715022,-2176.085027,2406.037606,1


In [ ]:
res2 = grid_cv2.cv_results_
res2 = pd.DataFrame(res2)
#res2
res2.columns

res2[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr2__n_neighbors', 'param_knr2__p', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error', 'split0_test_recall',
       'split1_test_recall', 'split2_test_recall', 'split3_test_recall',
       'split4_test_recall', 'mean_test_recall', 'std_test_recall',
       'rank_test_recall', 'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error']].sort_values('rank_test_neg_mean_squared_error')


#Note that best param {knr2__n_neighbors: 9,knr__p:2}, avg Mean -2221.6

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr2__n_neighbors,param_knr2__p,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_recall,rank_test_recall,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
7,0.012130,0.002313,0.012656,0.002416,9,2,"{'knr2__n_neighbors': 9, 'knr2__p': 2}",-14.929786,-14.416827,-16.194659,...,NaN,1,-1539.203920,-633.054250,-998.444498,-6948.007445,-761.715022,-2176.085027,2406.037606,1
5,0.010252,0.001006,0.011805,0.001767,7,2,"{'knr2__n_neighbors': 7, 'knr2__p': 2}",-14.828640,-14.400258,-16.855938,...,NaN,1,-1555.695932,-650.224353,-1157.582064,-6942.610238,-821.009093,-2225.424336,2378.835986,2
6,0.010959,0.003679,0.012200,0.003272,9,1,"{'knr2__n_neighbors': 9, 'knr2__p': 1}",-17.043467,-17.011486,-18.010067,...,NaN,1,-1607.519929,-900.159146,-1138.703982,-7017.940723,-745.236204,-2281.911997,2385.888126,3
3,0.012419,0.002940,0.014312,0.002426,5,2,"{'knr2__n_neighbors': 5, 'knr2__p': 2}",-16.396289,-16.562771,-14.194145,...,NaN,1,-1949.759575,-966.609921,-847.609792,-7318.201060,-1057.548943,-2427.945858,2476.053738,4
4,0.011038,0.003338,0.012942,0.003221,7,1,"{'knr2__n_neighbors': 7, 'knr2__p': 1}",-18.786902,-16.471377,-18.522444,...,NaN,1,-2111.917956,-749.030636,-1331.544077,-7317.116646,-849.013322,-2471.724527,2470.148292,5
1,0.010134,0.001705,0.015424,0.003021,3,2,"{'knr2__n_neighbors': 3, 'knr2__p': 2}",-14.987108,-20.503936,-13.972450,...,NaN,1,-1833.918297,-2112.196932,-856.201301,-7236.719594,-1079.006904,-2623.608606,2352.677992,6
2,0.011536,0.004691,0.013722,0.002413,5,1,"{'knr2__n_neighbors': 5, 'knr2__p': 1}",-19.651831,-18.097277,-18.797783,...,NaN,1,-2363.566108,-1200.156069,-1393.447609,-7513.018880,-700.535121,-2634.144757,2498.444504,7
0,0.013970,0.006551,0.013282,0.002475,3,1,"{'knr2__n_neighbors': 3, 'knr2__p': 1}",-17.627349,-19.975984,-19.183454,...,NaN,1,-1939.175732,-2020.643700,-1741.021132,-7663.872051,-978.036767,-2868.549876,2425.825914,8


In [20]:
param_grid3 = {
    'knr3__max_depth': [2, 3, 4,5],
    'knr3__min_samples_leaf': [1,2]
    }

grid_cv3 = GridSearchCV(
    estimator=pipe_knr3, 
    param_grid=param_grid3, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")

grid_cv3.fit(X_train, Y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess3',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transfomer',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('knr3', GradientBoostingRegressor())]),
             param_grid={'knr3__max_depth': [2, 3, 4, 5],
                         'knr3__min_samples_leaf': [1, 2]},
             refit='neg_mean_squared_error',
             scoring=['neg_mean_absolute_error', 'neg_mean_squared_error'])

In [21]:
res3 = grid_cv3.cv_results_
res3 = pd.DataFrame(res3)
res3
#res3.columns

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr3__max_depth,param_knr3__min_samples_leaf,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
0,0.489330,0.093461,0.032926,0.024201,2,1,"{'knr3__max_depth': 2, 'knr3__min_samples_leaf...",-23.181748,-32.873388,-15.993107,...,6.853567,4,-6486.222464,-6786.122118,-916.604606,-7337.096559,-806.448071,-4466.498764,2956.281030,6
1,0.265134,0.021893,0.015802,0.008698,2,2,"{'knr3__max_depth': 2, 'knr3__min_samples_leaf...",-20.056991,-27.859200,-15.607263,...,5.582221,1,-2445.128319,-3626.614058,-951.543093,-7492.279945,-653.965930,-3033.906269,2472.869194,1
2,0.357337,0.044978,0.017637,0.004523,3,1,"{'knr3__max_depth': 3, 'knr3__min_samples_leaf...",-23.044513,-28.460096,-22.740740,...,5.499696,8,-6313.130286,-6076.522679,-2656.134796,-7696.442029,-749.437832,-4698.333524,2579.146424,8
3,0.284908,0.015078,0.015563,0.005507,3,2,"{'knr3__max_depth': 3, 'knr3__min_samples_leaf...",-22.267777,-26.831357,-19.384579,...,4.837845,2,-3051.750409,-3168.012637,-1265.422370,-7385.752820,-773.513544,-3128.890356,2330.179050,3
4,0.385384,0.117383,0.013550,0.003292,4,1,"{'knr3__max_depth': 4, 'knr3__min_samples_leaf...",-21.954050,-29.536285,-18.699357,...,6.097155,7,-3814.330153,-6233.704873,-1283.639762,-8143.522376,-831.715142,-4061.382461,2813.911558,5
5,0.320008,0.015942,0.013502,0.005064,4,2,"{'knr3__max_depth': 4, 'knr3__min_samples_leaf...",-22.656819,-29.229744,-18.821365,...,5.428889,3,-3460.691763,-3579.437590,-1167.938684,-7216.071628,-788.931673,-3242.614268,2292.344136,4
6,0.386038,0.044217,0.018891,0.005622,5,1,"{'knr3__max_depth': 5, 'knr3__min_samples_leaf...",-23.539488,-29.126053,-17.547315,...,6.168849,6,-5482.752928,-7552.517121,-949.070674,-7787.861644,-755.666617,-4505.573797,3089.423069,7
7,0.505786,0.037792,0.023356,0.007455,5,2,"{'knr3__max_depth': 5, 'knr3__min_samples_leaf...",-21.475948,-30.295563,-20.380372,...,5.366564,5,-2965.703410,-3024.351970,-1235.417952,-7497.660756,-843.945839,-3113.415985,2363.418074,2


In [22]:
res3[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr3__max_depth', 'param_knr3__min_samples_leaf', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error', 'split0_test_recall',
       'split1_test_recall', 'split2_test_recall', 'split3_test_recall',
       'split4_test_recall', 'mean_test_recall', 'std_test_recall',
       'rank_test_recall', 'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error']].sort_values('rank_test_neg_mean_squared_error')

#Note that best param {'knr3__max_depth': 2, 'knr3__min_samples_leaf': 2}, avg mean = -3037.6

KeyError: "['split0_test_recall', 'split1_test_recall', 'split2_test_recall', 'split3_test_recall', 'split4_test_recall', 'mean_test_recall', 'std_test_recall', 'rank_test_recall'] not in index"

In [23]:
Best_param3 = grid_cv3.best_params_
#Best_param1 = pd.DataFrame(Best_param1)
print(Best_param3)
print(grid_cv3.best_estimator_)

{'knr3__max_depth': 2, 'knr3__min_samples_leaf': 2}
Pipeline(steps=[('preprocess3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('knr3',
                 GradientBoostingRegressor(max_depth=2, min_samples_leaf=2))])


In [24]:
param_grid4 = {
    'knr4__max_depth': [2, 3, 4,5],
    'knr4__min_samples_leaf': [1,2]
    }

grid_cv4 = GridSearchCV(
    estimator=pipe_knr4, 
    param_grid=param_grid4, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")

grid_cv4.fit(X_train, Y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess4',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transfomer',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('Power_transfomer',
                                                                         FunctionTransformer(),
                                                                         ['temp']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('knr4', GradientBoostingRegressor())]),
             param_grid={'knr4__max_depth': [2, 3, 4, 5],
                         'knr4__min_samples_leaf': [1, 2]},
             refit='neg_mean_squared_error',
             scoring=['neg_mean_absolute_error', 'neg_mean_squared_error'])

In [25]:
res4 = grid_cv4.cv_results_
res4 = pd.DataFrame(res4)
res4
#res4.columns

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr4__max_depth,param_knr4__min_samples_leaf,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
0,0.292771,0.053027,0.022775,0.003023,2,1,"{'knr4__max_depth': 2, 'knr4__min_samples_leaf...",-23.181748,-30.244956,-15.897691,...,6.060558,3,-6482.460768,-5227.576073,-911.879632,-7352.490934,-805.791831,-4156.039847,2775.840810,6
1,0.320968,0.061159,0.018991,0.004233,2,2,"{'knr4__max_depth': 2, 'knr4__min_samples_leaf...",-20.062885,-26.816752,-15.710172,...,5.238873,1,-2447.418581,-3232.304484,-961.665048,-7507.973213,-658.683248,-2961.608915,2462.126651,1
2,0.348014,0.014069,0.016448,0.004677,3,1,"{'knr4__max_depth': 3, 'knr4__min_samples_leaf...",-21.543508,-27.653160,-22.250824,...,4.938641,5,-4770.181298,-5460.044727,-2477.701356,-7696.158353,-916.327231,-4264.082593,2360.687579,7
3,0.306993,0.014771,0.017395,0.006061,3,2,"{'knr4__max_depth': 3, 'knr4__min_samples_leaf...",-22.560927,-27.577025,-19.497887,...,5.263910,2,-3057.565430,-3326.396173,-1270.606270,-7401.950952,-703.436223,-3151.991010,2351.021988,2
4,0.446897,0.084565,0.029492,0.015528,4,1,"{'knr4__max_depth': 4, 'knr4__min_samples_leaf...",-21.777472,-29.987538,-17.901812,...,6.513581,7,-3799.563080,-6545.023285,-1140.773088,-8117.331156,-730.846539,-4066.707430,2908.952657,5
5,0.441700,0.120422,0.022043,0.009662,4,2,"{'knr4__max_depth': 4, 'knr4__min_samples_leaf...",-22.522287,-29.055226,-19.898301,...,4.995335,4,-3453.116002,-3459.885050,-1292.608123,-7206.471364,-894.517742,-3261.319656,2241.339441,4
6,0.421266,0.039799,0.012942,0.003439,5,1,"{'knr4__max_depth': 5, 'knr4__min_samples_leaf...",-23.955225,-29.366789,-19.709927,...,5.361029,8,-6194.951410,-8489.043331,-1158.298659,-8009.779442,-1372.654608,-5044.945490,3180.135464,8
7,0.401365,0.012320,0.018423,0.006326,5,2,"{'knr4__max_depth': 5, 'knr4__min_samples_leaf...",-21.648882,-30.154826,-20.887495,...,5.306891,6,-3112.508358,-3055.208015,-1310.778682,-7500.464448,-867.971243,-3169.386149,2346.292533,3


In [ ]:
res4[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_knr4__max_depth', 'param_knr4__min_samples_leaf', 'params',
       'split0_test_neg_mean_absolute_error',
       'split1_test_neg_mean_absolute_error',
       'split2_test_neg_mean_absolute_error',
       'split3_test_neg_mean_absolute_error',
       'split4_test_neg_mean_absolute_error',
       'mean_test_neg_mean_absolute_error', 'std_test_neg_mean_absolute_error',
       'rank_test_neg_mean_absolute_error', 'split0_test_recall',
       'split1_test_recall', 'split2_test_recall', 'split3_test_recall',
       'split4_test_recall', 'mean_test_recall', 'std_test_recall',
       'rank_test_recall', 'split0_test_neg_mean_squared_error',
       'split1_test_neg_mean_squared_error',
       'split2_test_neg_mean_squared_error',
       'split3_test_neg_mean_squared_error',
       'split4_test_neg_mean_squared_error',
       'mean_test_neg_mean_squared_error', 'std_test_neg_mean_squared_error',
       'rank_test_neg_mean_squared_error']].sort_values('rank_test_neg_mean_squared_error')

#Note that best param {'knr3__max_depth': 2, 'knr3__min_samples_leaf': 2}, avg mean = -2986.5

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knr4__max_depth,param_knr4__min_samples_leaf,params,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,...,std_test_recall,rank_test_recall,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,split4_test_neg_mean_squared_error,mean_test_neg_mean_squared_error,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error
1,0.151135,0.019680,0.012800,0.003396,2,2,"{'knr4__max_depth': 2, 'knr4__min_samples_leaf...",-20.062885,-27.249759,-15.663252,...,NaN,1,-2447.418581,-3380.666282,-953.246808,-7497.529181,-653.731239,-2986.518418,2464.600931,1
7,0.221813,0.021741,0.009346,0.002021,5,2,"{'knr4__max_depth': 5, 'knr4__min_samples_leaf...",-21.950282,-29.773684,-20.602445,...,NaN,1,-3122.853217,-3064.409092,-1253.311919,-7351.519444,-880.490074,-3134.516749,2298.161470,2
3,0.160496,0.006941,0.011530,0.002678,3,2,"{'knr4__max_depth': 3, 'knr4__min_samples_leaf...",-22.381232,-27.526980,-19.584020,...,NaN,1,-3078.331058,-3218.620461,-1266.549933,-7386.839743,-772.219607,-3144.512160,2330.739341,3
5,0.211299,0.012992,0.014037,0.004782,4,2,"{'knr4__max_depth': 4, 'knr4__min_samples_leaf...",-22.467192,-28.638791,-19.179387,...,NaN,1,-3542.860245,-3367.119891,-1210.216466,-7199.928267,-785.840367,-3221.193047,2277.282058,4
0,0.191476,0.086315,0.017130,0.011570,2,1,"{'knr4__max_depth': 2, 'knr4__min_samples_leaf...",-23.194142,-30.995228,-15.898194,...,NaN,1,-6486.635672,-5709.020827,-911.900592,-7336.277696,-817.189754,-4252.204908,2813.661687,5
2,0.166986,0.010597,0.011612,0.004879,3,1,"{'knr4__max_depth': 3, 'knr4__min_samples_leaf...",-21.209838,-27.238525,-23.738063,...,NaN,1,-4284.908182,-5650.862739,-2873.330113,-7665.595670,-888.767093,-4272.692760,2316.050975,6
4,0.254268,0.068537,0.011980,0.006952,4,1,"{'knr4__max_depth': 4, 'knr4__min_samples_leaf...",-22.232462,-31.553584,-18.709847,...,NaN,1,-4049.975309,-7140.384320,-1221.585925,-8117.106568,-1152.614715,-4336.333368,2900.919329,7
6,0.268375,0.020613,0.012758,0.003093,5,1,"{'knr4__max_depth': 5, 'knr4__min_samples_leaf...",-24.029080,-28.108985,-17.244462,...,NaN,1,-6299.900982,-6689.927408,-857.202823,-8067.204679,-841.127516,-4551.072682,3079.111945,8


# Evaluate

+ Which model has the best performance?

** Pipeline A = preproc1 + baseline has the best mean_test_neg_mean_squared_error.

# Export

+ Save the best performing model to a pickle file.

In [26]:
import pickle 
filename = 'model_pipe_knr.pkl'
with open(filename, 'wb') as file:
        pickle.dump(pipe_knr, file)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [27]:
import shap

In [31]:
# Not able to get Shap to work with KNeighborsRegressor
#import shap
pipe_knr.fit(X_train, Y_train)
data_transform = pipe_knr.named_steps['preprocess'].transform(X_test)

explainer = shap.Explainer(
    pipe_knr.named_steps['knr'], 
    data_transform,
    feature_names = pipe_knr.named_steps['preprocess'].get_feature_names_out())

shap_values = explainer(data_transform)


TypeError: The passed model is not callable and cannot be analyzed directly with the given masker! Model: KNeighborsRegressor()

In [ ]:
#shap.plots.waterfall(shap_values[1])

In [ ]:
#shap.plots.beeswarm(shap_values)

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.